# Nonlinear static analysis with structsolve

`structsolve.Analysis`, of
[structsolve](https://saullocastro.github.io/structsolve/), solves
nonlinear static problems from four callables written by the user:
`calc_fext`, `calc_fint`, `calc_kC` and `calc_kG`. This example shows the
callables of a model of `bfsccylinder` elements, see the section Solving
with structsolve of [Analyses](../analyses.rst), first for dead loads and
then for a follower pressure, whose load stiffness matrix is unsymmetric.
The callables are those of the tests
`tests/test_nonlinear_axial_compression.py` and
`tests/test_follower_pressure_structsolve.py`, imported from there.

In [1]:
import inspect
import os
import sys
import numpy as np
from numpy import isclose
import matplotlib.pyplot as plt
from composites import isotropic_plate
from structsolve import Analysis
from structsolve.convergence import crisfield_weights, crisfield_zeta

sys.path.insert(0, os.path.join('..', 'tests'))
from test_nonlinear_axial_compression import Callables
from test_follower_pressure_structsolve import FollowerCallables
from test_follower_pressure_validation import KINEMATICS, E, NU, cylinder_model

from IPython.display import Markdown


def table(rows, fmt='{:.5g}'):
    """Markdown table of a list of dicts with the same keys"""
    keys = list(rows[0])
    cell = lambda v: fmt.format(v) if isinstance(v, float) else str(v)
    lines = ['| ' + ' | '.join(keys) + ' |', '|' + '---|'*len(keys)]
    lines += ['| ' + ' | '.join(cell(r[k]) for k in keys) + ' |' for r in rows]
    return Markdown('\n'.join(lines))

## Callables for dead loads

The callables work on the unknown degrees of freedom `bu`, the partition
of the free degrees of freedom of structsolve: the solution vector of the
solvers is $\boldsymbol{c} = \boldsymbol{u}_u$, the prescribed degrees of
freedom being zero, and `expand` gives the displacements of the whole
model. `calc_fint` assembles the internal forces of `update_fint`;
`calc_kC` gives $\boldsymbol{K}_{C0}$, or, with `NLgeom=True`,
$\boldsymbol{K}_{C0} + \boldsymbol{K}_{CNL} + \boldsymbol{K}_{GNL}$ of
`update_KCNL`; `calc_kG` gives $\boldsymbol{K}_G$ of `update_KG`. Their sum
is the tangent stiffness matrix $\boldsymbol{K}_T$, the exact Jacobian of
the residual $\boldsymbol{F}_{int}(\boldsymbol{u}) - \lambda
\boldsymbol{F}_{ext}$, and the value arrays of the sparse matrices are
zeroed in each call, since the update functions add to them:

In [2]:
print(inspect.getsource(Callables))

class Callables(object):
    """The callables of structsolve.Analysis for the ``elements`` of a model
    under the external forces ``fext``.

    They loop over the elements with the update functions and assemble, and
    work on the unknown degrees of freedom ``bu`` only, the partition of the
    free dofs of structsolve: the solution vector of the solvers is ``c =
    u[bu]``, and the force vectors and matrices are restricted to ``bu``.
    The prescribed degrees of freedom are zero.
    """
    def __init__(self, elements, points, weights, KC0, bu, fext):
        self.elements = elements
        self.points = points
        self.weights = weights
        self.N = KC0.shape[0]
        self.KC0 = KC0
        self.KC0uu = KC0[bu, :][:, bu]
        self.bu = bu
        self.fext = fext
        num_elements = len(elements)
        self.KCNLr = np.zeros(KCNL_SPARSE_SIZE*num_elements, dtype=INT)
        self.KCNLc = np.zeros(KCNL_SPARSE_SIZE*num_elements, dtype=INT)
        self.KCNLv = n

[Nonlinear static analysis](ex_nonlinear_static.ipynb) solves a laminated
cylinder under axial compression with these callables.

## Callables for a follower pressure

A pressure normal to the deformed shell depends on the displacements. With
the follower pressure $\lambda p_0$ the residual and its Jacobian are, see
[Follower pressure](../follower_pressure.rst),

$$
\boldsymbol{R}(\boldsymbol{u}) = \boldsymbol{F}_{int}(\boldsymbol{u}) -
\lambda \boldsymbol{F}_p(\boldsymbol{u}), \qquad
\boldsymbol{K}_T = \boldsymbol{K}_{C0} + \boldsymbol{K}_{CNL} +
\boldsymbol{K}_{GNL} + \boldsymbol{K}_G + \boldsymbol{K}_F
$$

The callables receive the load factor `inc`: `calc_fext` gives the dead
part $\lambda \boldsymbol{F}_p(\boldsymbol{0})$, `calc_fint` subtracts the
part that depends on the displacements, $\lambda
(\boldsymbol{F}_p(\boldsymbol{u}) - \boldsymbol{F}_p(\boldsymbol{0}))$, of
`update_fext_follower` with `reference=0`, and `calc_kC` adds the load
stiffness matrix $\boldsymbol{K}_F$ of $\lambda p_0$ of
`update_KCfollower`. They also record the iterations, to follow the
convergence:

In [3]:
print(inspect.getsource(FollowerCallables))

class FollowerCallables(object):
    r"""The callables of structsolve.Analysis for the ``elements`` of the
    module ``mod`` under the follower pressures ``pressures``, one array of
    shape ``(nint, nint)`` per element, times the load factor

    They loop over the elements with the update functions and assemble, and
    work on the unknown degrees of freedom ``bu`` only, the partition of the
    free dofs of structsolve: the solution vector of the solvers is ``c =
    u[bu]``, and the force vectors and matrices are restricted to ``bu``.
    ``quadratic`` is the truncation of the area vector. ``calc_fint`` also
    keeps the load factor and the internal forces of each iteration, for the
    residual history.
    """
    def __init__(self, mod, elements, points, weights, N, bu, pressures,
                 quadratic=0):
        self.mod = mod
        self.elements = elements
        self.points = points
        self.weights = weights
        self.N = N
        self.bu = bu
        sel

### Clamped-free cylinder

$R = 1$ m, $L = 1$ m and $h = 10$ mm, isotropic, with the $E$ and $\nu$ of
`tests/test_follower_pressure_validation.py`, whose `cylinder_model`
builds the mesh of 7 by 24 nodes and the elements of either kinematics. All
ten degrees of freedom are constrained at the nodes of $x = 0$; the end
$x = L$ is free. The external pressure
$p = -2 (1 + \cos(3 y/R)/2)$ bar varies around the circumference and is
given at the integration points of each element. The load stiffness matrix
is symmetric only if $p$ is uniform and, along the contour of the loaded
region, $w = 0$ or the in-plane displacement normal to the contour
vanishes [Hibbitt1979], [SchweizerhofRamm1984]: here both conditions fail.

In [4]:
R, L, h = 1., 1., 0.01  # m
prop = isotropic_plate(thickness=h, E=E, nu=NU)
print(f'E = {E/1e9} GPa, nu = {NU}')

def pressures_of(elements, y, points):
    """p at the integration points of each element, shape (nint, nint)"""
    nint = len(points)
    out = []
    for shell in elements:
        yj = y[shell.n1] + shell.ley*(points + 1)/2
        p = -2.e5*(1 + 0.5*np.cos(3*yj/R))  # Pa
        out.append(np.ascontiguousarray(np.ones((nint, 1))*p[None, :]))
    return out

E = 70.0 GPa, nu = 0.3


The analysis, for both kinematics, in two equal increments of the load
factor, 0.5 and 1. After it, the equilibrium with the whole follower
pressure is checked with $\boldsymbol{F}_p$ at the final displacements, the
asymmetry of $\boldsymbol{K}_F$ is measured, and the convergence history of
each increment is computed with Crisfield's criterion $\zeta$
[Crisfield1981], the criterion of structsolve.

In [5]:
results = {}
for kinematics in sorted(KINEMATICS):
    mod, elements, points, weights, x, y, N = cylinder_model(
        kinematics, L, R, prop, nx=7, ny=24)
    DOF = mod.DOF
    bk = np.zeros(N, dtype=bool)
    for i in range(DOF):
        bk[i::DOF] |= isclose(x, 0)
    bu = ~bk
    pressures = pressures_of(elements, y, points)
    callables = FollowerCallables(mod, elements, points, weights, N, bu,
                                  pressures)
    an = Analysis(calc_fext=callables.calc_fext, calc_fint=callables.calc_fint,
                  calc_kC=callables.calc_kC, calc_kG=callables.calc_kG)
    an.initialInc = 0.5
    an.maxInc = 0.5
    increments, cs = an.static(NLgeom=True)
    u = callables.expand(cs[-1])

    fint = np.zeros(N)
    fp = np.zeros(N)
    for shell, p in zip(elements, pressures):
        mod.update_fint(u, shell, points, weights, fint)
        mod.update_fext_follower(u, shell, points, weights, p, fp)
    residual = np.linalg.norm((fint - fp)[bu])/np.linalg.norm(fp[bu])
    follower = (np.linalg.norm((fp - callables.fp0)[bu])
                /np.linalg.norm(callables.fp0[bu]))

    KCf = (callables.calc_kC(cs[-1], NLgeom=True, inc=1.)
           - callables.calc_kC(cs[-1], NLgeom=True, inc=0.)).toarray()
    asym = np.abs(KCf - KCf.T).max()/np.abs(KCf).max()

    wts = crisfield_weights(callables.KC0[bu, :][:, bu])
    f = callables.fp0[bu]
    zeta = {lbd: [crisfield_zeta(inc*f - fi, inc*f, fi, wts)
                  for inc, fi in callables.iterations if inc == lbd]
            for lbd in increments}
    results[kinematics] = dict(increments=increments, u=u, x=x, y=y, N=N,
                               DOF=DOF, wmax=np.abs(u[6::DOF]).max()/h,
                               residual=residual, follower=follower,
                               asym=asym, zeta=zeta)

Started Non-Linear Static Analysis
	___________________________________________
	                                           
	Newton-Raphson solver
	___________________________________________
	Initializing...
	Step 1, attempting load factor 0.50000
		Iteration: 1, max(|R|) = 9.725e+02, zeta = 2.968e-02


		Iteration: 2, max(|R|) = 3.290e+00, zeta = 1.612e-04


		Iteration: 3, max(|R|) = 2.833e-04, zeta = 1.715e-08


		Iteration: 4, max(|R|) = 7.584e-11, zeta = 1.374e-15
		Converged at load factor 0.50000
	Step 2, attempting load factor 1.00000
		Iteration: 1, max(|R|) = 1.989e+03, zeta = 3.117e-02


		Iteration: 2, max(|R|) = 4.744e+01, zeta = 1.353e-03


		Iteration: 3, max(|R|) = 7.309e-02, zeta = 2.064e-06


... (5 outputs left out)

		Iteration: 4, max(|R|) = 9.108e-11, zeta = 1.472e-15
		Converged at load factor 0.50000
	Step 2, attempting load factor 1.00000
		Iteration: 1, max(|R|) = 1.972e+03, zeta = 2.536e-02


		Iteration: 2, max(|R|) = 3.198e+01, zeta = 9.283e-04


		Iteration: 3, max(|R|) = 3.122e-02, zeta = 8.804e-07


		Iteration: 4, max(|R|) = 9.291e-08, zeta = 2.170e-12
		Converged at load factor 1.00000
Finished Non-Linear Static Analysis
	    load factor 1.00000


### Results

Both increments converged without a cut. The table gives, for each
kinematics, the largest $|w|/h$, the relative residual with the whole
follower pressure, the relative size of the follower part of the load,
$\| \boldsymbol{F}_p(\boldsymbol{u}) - \boldsymbol{F}_p(\boldsymbol{0}) \|
/ \| \boldsymbol{F}_p(\boldsymbol{0}) \|$, and the asymmetry
$\max |\boldsymbol{K}_F - \boldsymbol{K}_F^\top| / \max |\boldsymbol{K}_F|$.
No published value exists for this case; the Sanders-type result is the
reference of the Donnell-type one.

In [6]:
ref = results['sanders']['wmax']
table([{'kinematics': k, 'load factors': r['increments'],
               'max abs(w)/h': r['wmax'],
               'rel. diff. to Sanders (%)': 100*(r['wmax'] - ref)/ref,
               'relative residual': r['residual'],
               'follower part': r['follower'],
               'asymmetry of K_F': r['asym']}
              for k, r in results.items()])

| kinematics | load factors | max abs(w)/h | rel. diff. to Sanders (%) | relative residual | follower part | asymmetry of K_F |
|---|---|---|---|---|---|---|
| donnell | [0.5, 1.0] | 0.39922 | 2.1375 | 2.5146e-11 | 0.0051691 | 2 |
| sanders | [0.5, 1.0] | 0.39087 | 0 | 4.7221e-12 | 0.0047277 | 2 |

Crisfield's $\zeta$ of each iteration: the last iteration of each increment
squares the previous value down to the round-off, the quadratic
convergence that only a tangent stiffness matrix equal to the Jacobian of
the residual, here unsymmetric, gives.

In [7]:
fig, ax = plt.subplots(figsize=(5, 3.5))
for k, r in results.items():
    for lbd, hist in r['zeta'].items():
        ax.semilogy(range(1, len(hist) + 1), hist, 'o-',
                    label=f'{k}, load factor {lbd}')
ax.axhline(1e-9, color='k', lw=0.8, ls='--')
ax.set_xlabel('iteration')
ax.set_ylabel(r'$\zeta$')
ax.legend(fontsize=8)
plt.show()

In [8]:
r = results['sanders']
w = r['u'][6::r['DOF']]
fig, ax = plt.subplots(figsize=(6, 3))
c = ax.tricontourf(r['y'], r['x'], w/h, levels=40, cmap='coolwarm')
fig.colorbar(c, ax=ax, label='$w/h$')
ax.set_xlabel('$y$ (m)')
ax.set_ylabel('$x$ (m)')
ax.set_title('Sanders-type kinematics, load factor 1')
plt.show()

## References

- **[Crisfield1981]** Crisfield, M. A., A fast incremental/iterative solution procedure that
  handles "snap-through", Computers & Structures, 13(1-3), 55-62, 1981,
  doi:10.1016/0045-7949(81)90108-5.
- **[Hibbitt1979]** Hibbitt, H. D., Some follower forces and load stiffness, International
  Journal for Numerical Methods in Engineering, 14(6), 937-941, 1979,
  doi:10.1002/nme.1620140613.
- **[SchweizerhofRamm1984]** Schweizerhof, K., and Ramm, E., Displacement dependent pressure loads in
  nonlinear finite element analyses, Computers & Structures, 18(6),
  1099-1114, 1984, doi:10.1016/0045-7949(84)90154-8.